# Εργαστήριο: Χρήση της Οντολογίας Πανεπιστημίου μέσω Python

Αυτό το notebook συνοδεύει τον εργαστηριακό οδηγό του Protégé. Περιέχει όλο τον απαραίτητο κώδικα για τη φόρτωση, την εξερεύνηση, την εκτέλεση του Reasoner και τη χρήση ερωτημάτων SPARQL πάνω στην οντολογία `university.owl` με τη βοήθεια της βιβλιοθήκης `owlready2`.

## 0. Εγκατάσταση Απαιτούμενων Βιβλιοθηκών
Εγκαθιστούμε την `owlready2`. *Σημείωση: Για να εκτελεστεί ο Reasoner στο βήμα 5, βεβαιωθείτε ότι έχετε εγκατεστημένη τη **Java** στο σύστημά σας.*

In [ ]:
!pip install owlready2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 9.8 MB/s  0:00:026m0:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for owlready2: filename=owlready2-0.50-cp314-cp314-linux_x86_64.whl size=24667607 sha256=699e0c5988f8d43079290bde88e3a0006e549924062be413d09e26ee040c40f9
  Stored in directory: /home/rg/.cache/pip/wheels/d2/2c/e6/6ccb6639e720c2611d083a0ca1eb4d0f9473de7399688b343e
Successfully built owlready2

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 1. Φόρτωση της Οντολογίας
Φορτώνουμε το αρχείο `university.owl` (το οποίο πρέπει να έχετε εξάγει από το Protégé και να βρίσκεται στον ίδιο φάκελο με αυτό το notebook).

In [ ]:
from owlready2 import get_ontology

# Φόρτωση αρχείου OWL (αν το αρχείο είναι σε άλλο φάκελο, προσαρμόστε το path)
onto = get_ontology("file://protege/university.owx").load()

print("IRI οντολογίας:", onto.base_iri)

IRI οντολογίας: http://www.example.org/university#


## 2. Εξερεύνηση Κλάσεων και Ιδιοτήτων
Μπορούμε να δούμε δυναμικά όλα τα δομικά στοιχεία (T-Box) που έχουμε σχεδιάσει στο γραφικό περιβάλλον του Protégé.

In [ ]:
# Εκτύπωση όλων των κλάσεων
print("=== Κλάσεις ===")
for cls in onto.classes():
    print(" -", cls.name)

# Εκτύπωση όλων των object properties
print("\n=== Object Properties ===")
for prop in onto.object_properties():
    print(" -", prop.name)

# Εκτύπωση όλων των data properties
print("\n=== Data Properties ===")
for prop in onto.data_properties():
    print(" -", prop.name)

=== Κλάσεις ===
 - AdvancedCourse
 - Course
 - Department
 - IntroductoryCourse
 - Professor

=== Object Properties ===
 - hasPrerequisite
 - offeredBy
 - taughtBy

=== Data Properties ===
 - courseCode
 - credits
 - departmentName
 - semester


## 3. Εξερεύνηση Individuals (Στιγμιότυπα)
Εδώ ανακτούμε τα δεδομένα (A-Box), δηλαδή τα συγκεκριμένα μαθήματα που περάσαμε στην οντολογία.

In [ ]:
print("=== Μαθήματα ===")
for course in onto.Course.instances():
    # Επειδή τα data properties επιστρέφουν πάντα λίστες, παίρνουμε το 1ο στοιχείο (αν υπάρχει)
    code = course.courseCode[0] if course.courseCode else "—"
    cr   = course.credits[0]    if course.credits    else "—"
    sem  = course.semester[0]   if course.semester   else "—"
    
    print(f"  {course.name:20s}  κωδικός={code}  credits={cr}  εξάμηνο={sem}")

=== Μαθήματα ===
  Algorithms            κωδικός=CS301  credits=6  εξάμηνο=5
  DataStructures        κωδικός=CS201  credits=6  εξάμηνο=3
  MachineLearning       κωδικός=CS401  credits=6  εξάμηνο=5
  Math1                 κωδικός=MAT101  credits=5.0  εξάμηνο=1.0
  Programming1          κωδικός=CS101  credits=5  εξάμηνο=1


## 4. Ερώτηση Προαπαιτουμένων (Άμεσα)
Ζητάμε να δούμε ποια μαθήματα είναι *ρητά* δηλωμένα ως προαπαιτούμενα για το `MachineLearning`.

In [ ]:
ml = onto.search_one(iri="*MachineLearning")

print(f"Άμεσα προαπαιτούμενα του {ml.name}:")
for prereq in ml.hasPrerequisite:
    print(" -", prereq.name)

Άμεσα προαπαιτούμενα του MachineLearning:
 - DataStructures
 - Math1


## 5. Εξαγωγή Έμμεσων Προαπαιτουμένων (Reasoner)
Στο Protégé ορίσαμε το `hasPrerequisite` ως **Transitive** (Μεταβατικό). Τρέχοντας τον Reasoner (εδώ τον HermiT) μέσα από την Python, η μηχανή θα υπολογίσει και θα συμπεράνει αυτόματα τα έμμεσα προαπαιτούμενα.

In [ ]:
from owlready2 import sync_reasoner_hermit

# Εκτέλεση του reasoner μέσα στο context της οντολογίας
print("Εκκίνηση του Reasoner (HermiT)...")
with onto:
    sync_reasoner_hermit(infer_property_values=True)
print("Ολοκληρώθηκε.\n")

# Τώρα βλέπουμε ΚΑΙ τις έμμεσες (inferred) συνδέσεις!
print(f"Όλα τα προαπαιτούμενα (inferred) του {ml.name}:")
for prereq in ml.hasPrerequisite:
    print(" -", prereq.name)

Εκκίνηση του Reasoner (HermiT)...


* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /home/rg/Teaching/uniwa/dss-data-analytics/lab1venv/lib/python3.14/site-packages/owlready2/hermit:/home/rg/Teaching/uniwa/dss-data-analytics/lab1venv/lib/python3.14/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////tmp/tmpa7of67of -Y


* Owlready * Adding relation university.owx.MachineLearning hasPrerequisite university.owx.Programming1
* Owlready * Adding relation university.owx.Algorithms hasPrerequisite university.owx.Programming1
Ολοκληρώθηκε.

Όλα τα προαπαιτούμενα (inferred) του MachineLearning:
 - DataStructures
 - Math1
 - Programming1


* Owlready2 * HermiT took 1.0376179218292236 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


## 6. Εκτέλεση Ερωτημάτων SPARQL
Η `owlready2` μας επιτρέπει να τρέχουμε δομημένα SPARQL queries κατευθείαν στον RDF γράφο μας.

In [ ]:
from owlready2 import default_world

# Ερώτημα: Φέρε όλα τα Προχωρημένα Μαθήματα (AdvancedCourse) και τα προαπαιτούμενά τους
results = default_world.sparql("""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX onto: <http://www.example.org/university#>

    SELECT ?course ?prereq
    WHERE {
        ?course rdf:type onto:AdvancedCourse .
        ?course onto:hasPrerequisite ?prereq .
    }
    ORDER BY ?course
""")

print("AdvancedCourse → Προαπαιτούμενο:")
for row in results:
    print(f"  {row[0].name:20s} → {row[1].name}")

AdvancedCourse → Προαπαιτούμενο:
  Algorithms           → DataStructures
  Algorithms           → Programming1
  DataStructures       → Programming1
  MachineLearning      → DataStructures
  MachineLearning      → Math1
  MachineLearning      → Programming1


## 7. Δυναμική Προσθήκη Νέας Γνώσης
Μπορούμε να δημιουργήσουμε δυναμικά νέα μαθήματα μέσω κώδικα και να αποθηκεύσουμε την τροποποιημένη οντολογία πίσω στον δίσκο μας.

In [8]:
with onto:
    # 1. Δημιουργία στιγμιότυπου της κλάσης AdvancedCourse
    new_course = onto.AdvancedCourse("DeepLearning")
    
    # 2. Εισαγωγή Data Properties (πάντα περνιούνται ως λίστες)
    new_course.courseCode = ["CS501"]
    new_course.credits    = [6]
    new_course.semester   = [9]
    
    # 3. Εισαγωγή Object Properties (το MachineLearning είναι προαπαιτούμενο)
    new_course.hasPrerequisite.append(ml)

print("Προστέθηκε το μάθημα:", new_course.name)
print("Με προαπαιτούμενο:", new_course.hasPrerequisite[0].name)

# Αποθήκευση της νέας οντολογίας σε μορφή XML/RDF
onto.save(file="university_updated.owl", format="rdfxml")
print("\nΑποθηκεύτηκε επιτυχώς ως 'university_updated.owl'")

Προστέθηκε το μάθημα: DeepLearning
Με προαπαιτούμενο: MachineLearning

Αποθηκεύτηκε επιτυχώς ως 'university_updated.owl'
